In [ ]:
# Setup: load env, create v1 client
import os
from dotenv import load_dotenv
from md_python import MDClient, Experiment, ExperimentDesign, SampleMetadata

load_dotenv()
api_token = os.getenv("MD_AUTH_TOKEN")
assert api_token, "MD_AUTH_TOKEN not set (check your .env)"
base_url = os.getenv("MD_API_BASE_URL")  # optional

client = MDClient(api_token=api_token, base_url=base_url, version="v1")
print("V1 client initialized.")

In [ ]:
# Quick connectivity test
health = client.health.check()
print("Health:", health)

In [ ]:
from pathlib import Path
from typing import Iterable, List, Optional

LOCAL_DATA_DIR = Path("/Users/giuseppeinfusini/wd/md-repos/md-python/development/GI/do_not_save/test_data").resolve()
assert LOCAL_DATA_DIR.exists(), f"Path not found: {LOCAL_DATA_DIR}"

OVERRIDE_FILENAMES = [
    "proteomics_proteins_COMBINED.tsv",
    "proteomics_peptides_COMBINED.tsv",
]
filenames = OVERRIDE_FILENAMES
print("Files to upload:", filenames)

In [ ]:
import csv
from typing import Tuple

DESIGN_CSV_NAME = "experiment_design_COMBINED.csv"

def load_design_and_metadata_from_csv(csv_path: Path) -> Tuple[ExperimentDesign, SampleMetadata]:
    with open(csv_path, "r", encoding="utf-8") as f:
        raw = list(csv.reader(f))
    header = [h.strip() for h in raw[0]]
    hl = [h.lower() for h in header]
    idx_fn = next(i for i, h in enumerate(hl) if h in ("filename", "file"))
    idx_sample = next(i for i, h in enumerate(hl) if h in ("sample_name", "sample"))
    idx_cond = next(i for i, h in enumerate(hl) if h in ("condition", "group"))
    data_rows = raw[1:]

    design_data = [["filename", "sample_name", "condition"]]
    for r in data_rows:
        design_data.append([r[idx_fn].strip(), r[idx_sample].strip(), r[idx_cond].strip()])
    exp_design = ExperimentDesign(data=design_data)

    sample_col_indices = [i for i, h in enumerate(hl) if h not in ("filename", "file")]
    sample_headers = [header[i] for i in sample_col_indices]
    seen, sample_rows = set(), []
    for r in data_rows:
        sn = r[idx_sample].strip()
        if sn and sn not in seen:
            seen.add(sn)
            sample_rows.append([r[i].strip() for i in sample_col_indices])
    sample_md = SampleMetadata(data=[sample_headers] + sample_rows)
    return exp_design, sample_md

design_csv_path = LOCAL_DATA_DIR / DESIGN_CSV_NAME
exp_design, sample_md = load_design_and_metadata_from_csv(design_csv_path)
print(f"Design rows: {len(exp_design.data) - 1}, Sample metadata rows: {len(sample_md.data) - 1}")

In [ ]:
from datetime import datetime

experiment_name = "V1_test_experiment"
experiment_source = os.getenv("MD_DEFAULT_SOURCE", "md_format")
labelling_method = os.getenv("MD_LABELLING_METHOD", "lfq")

experiment = Experiment(
    name=experiment_name,
    source=experiment_source,
    labelling_method=labelling_method,
    experiment_design=exp_design,
    sample_metadata=sample_md,
    file_location=str(LOCAL_DATA_DIR),
    filenames=filenames,
)

experiment_id = client.experiments.create(experiment)
print("Experiment created. ID:", experiment_id)

In [ ]:
# Poll until experiment reaches a terminal state
# experiment_id = "..."  # uncomment and set if resuming
completed = client.experiments.wait_until_complete(experiment_id, poll_s=10, timeout_s=3600)
print(completed)

In [ ]:
# Get experiment by name or ID
exp_by_name = client.experiments.get_by_name(experiment_name)
print(exp_by_name)

exp_by_id = client.experiments.get_by_id(experiment_id)
print(exp_by_id)

In [ ]:
# Get the initial intensity dataset for the experiment
initial_dataset = client.datasets.find_initial_dataset(experiment_id)
print("Initial dataset ID:", initial_dataset.id)
initial_dataset

In [ ]:
# List all datasets for this experiment
datasets = client.datasets.list_by_experiment(experiment_id)
for ds in datasets:
    print(ds.id, ds.name, ds.state)

In [ ]:
from md_python import PairwiseComparisonDataset

condition_column = "condition"
control_condition = "md00001_a"

comparisons = sample_md.pairwise_vs_control(column=condition_column, control=control_condition)
print("Comparisons:", comparisons)

pw = PairwiseComparisonDataset(
    input_dataset_ids=[str(initial_dataset.id)],
    dataset_name="Pairwise V1 test",
    sample_metadata=sample_md,
    condition_column=condition_column,
    condition_comparisons=comparisons,
    entity_type="protein",
)
pairwise_dataset_id = pw.run(client)
print("Pairwise dataset ID:", pairwise_dataset_id)

In [ ]:
pw_state = client.datasets.wait_until_complete(
    experiment_id=str(experiment_id),
    dataset_id=str(pairwise_dataset_id),
    poll_s=10,
    timeout_s=3600,
)
print("Pairwise state:", pw_state.state)
pw_state